In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import logging
import re

# ============================================================

# Project paths

# ============================================================

In [ ]:
PROJECT_ROOT = Path(".")

RAW_DIR = PROJECT_ROOT / "Data_raw" / "UbiBrowser" / "Raw"

INTERIM_DIR = PROJECT_ROOT / "Data_interim" / "ubibrowser"

PROC_DIR = PROJECT_ROOT / "Data_proc" / "positives"

QC_DIR = PROJECT_ROOT / "Data_proc" / "qc_reports"

LOG_DIR = PROJECT_ROOT / "logs"

for d in [INTERIM_DIR, PROC_DIR, QC_DIR, LOG_DIR]:

    d.mkdir(parents=True, exist_ok=True)

LOG_FILE = LOG_DIR / "01_parse_ubibrowser_positives.log"

logging.basicConfig(

    filename=LOG_FILE,

    level=logging.INFO,

    format="%(asctime)s | %(levelname)s | %(message)s",

    force=True,

)

console = logging.StreamHandler()

console.setLevel(logging.INFO)

console.setFormatter(logging.Formatter("%(levelname)s | %(message)s"))

logging.getLogger("").addHandler(console)

E3_RAW_PATH = RAW_DIR / "E3-substrate-interactions.txt"

DUB_RAW_PATH = RAW_DIR / "DUB-substrate-interactions.txt"

# ============================================================

# Utility functions

# ============================================================

In [ ]:
def read_any_table(path: Path) -> pd.DataFrame:

    """

    Read a raw UbiBrowser table.

    Tries automatic separator detection first, then explicit separators.

    """

    if not path.exists():

        raise FileNotFoundError(f"File not found: {path}")

    attempts = [

        {"sep": None, "engine": "python"},

        {"sep": "\t", "engine": "python"},

        {"sep": ",", "engine": "python"},

        {"sep": ";", "engine": "python"},

    ]

    best_df = None

    best_ncols = 0

    best_params = None

    for params in attempts:

        try:

            df = pd.read_csv(path, dtype=str, **params)

            if df.shape[1] > best_ncols:

                best_df = df

                best_ncols = df.shape[1]

                best_params = params

        except Exception as e:

            logging.warning(f"Failed reading {path.name} with {params}: {e}")

    if best_df is None or best_df.shape[1] < 2:

        raise ValueError(f"Could not parse table: {path}")

    logging.info(f"Read {path.name}: shape={best_df.shape}, params={best_params}")

    return best_df

def clean_colname(col: str) -> str:

    """

    Normalize column names into lowercase snake_case.

    """

    col = str(col).replace("\ufeff", "").strip().lower()

    col = re.sub(r"[^a-z0-9]+", "_", col)

    col = re.sub(r"_+", "_", col).strip("_")

    return col

def normalize_ac(x):

    """

    Normalize UniProt accession.

    Removes isoform suffix.

    Example: P12345-2 -> P12345

    """

    if pd.isna(x):

        return np.nan

    x = str(x).strip()

    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:

        return np.nan

    x = re.split(r"[;,|]", x)[0].strip()

    x = x.split("-")[0].strip()

    return x

def normalize_gene(x):

    """

    Normalize gene symbol.

    Keeps the first symbol if multiple are present.

    """

    if pd.isna(x):

        return np.nan

    x = str(x).strip()

    if x == "" or x.lower() in {"nan", "none", "null", "-", "na", "n/a"}:

        return np.nan

    x = re.split(r"[;,|]", x)[0].strip()

    return x

def is_human_series(s: pd.Series) -> pd.Series:

    """

    Detect human species labels.

    """

    s = s.astype(str).str.strip().str.lower()

    return s.str.contains(

        r"homo\s*\.?\s*sapiens|h\.?\s*sapiens|human|h\s*\.?\s*sapien",

        regex=True,

        na=False,

    )

def filter_human_rows(df_raw: pd.DataFrame, enzyme_class: str) -> pd.DataFrame:

    """

    Filter UbiBrowser rows to human if species-related columns exist.

    If no species columns exist, returns unchanged table.

    """

    df = df_raw.copy()

    norm_cols = {c: clean_colname(c) for c in df.columns}

    species_cols = [

        c for c, cn in norm_cols.items()

        if cn in {"species", "organism", "organism_name"}

        or "species" in cn

        or "organism" in cn

    ]

    if species_cols:

        mask = pd.Series(False, index=df.index)

        for c in species_cols:

            mask = mask | is_human_series(df[c])

        before = len(df)

        df = df[mask].copy()

        logging.info(

            f"{enzyme_class}: human filter using species columns {species_cols}: "

            f"{before} -> {len(df)}"

        )

        return df

    # Fallback: check SwissProt ID columns with _HUMAN suffix

    id_cols = [c for c, cn in norm_cols.items() if "swissprot_id" in cn or "entry_name" in cn]

    if id_cols:

        mask = pd.Series(False, index=df.index)

        for c in id_cols:

            mask = mask | df[c].astype(str).str.upper().str.endswith("_HUMAN")

        before = len(df)

        df = df[mask].copy()

        logging.info(

            f"{enzyme_class}: human filter using SwissProt/entry columns {id_cols}: "

            f"{before} -> {len(df)}"

        )

        return df

    logging.warning(

        f"{enzyme_class}: no species or SwissProt ID column found. "

        f"No human filtering applied."

    )

    return df

def find_column(df: pd.DataFrame, candidates):

    """

    Find a column by normalized candidate names.

    """

    normalized_to_original = {clean_colname(c): c for c in df.columns}

    for cand in candidates:

        cand_clean = clean_colname(cand)

        if cand_clean in normalized_to_original:

            return normalized_to_original[cand_clean]

    return None

def standardize_positive_table(df_raw: pd.DataFrame, enzyme_class: str) -> pd.DataFrame:

    """

    Standardize E3 or DUB positive interaction table.

    Output schema:

    pair_id, group_id, enzyme_class, enz_ac, sub_ac,

    enz_gene, sub_gene, enzyme_type, label, source, pmid

    """

    df = df_raw.copy()

    logging.info(f"{enzyme_class}: raw columns = {list(df.columns)}")

    if enzyme_class == "E3":

        enz_ac_candidates = [

            "SwissProt AC (E3)",

            "UniProt AC (E3)",

            "E3 UniProt AC",

            "E3 SwissProt AC",

            "e3_uniprot_ac",

            "e3_ac",

        ]

        enz_gene_candidates = [

            "Gene Symbol (E3)",

            "E3 Gene Symbol",

            "E3",

            "e3_gene",

            "e3_symbol",

        ]

        enzyme_type_candidates = [

            "E3TYPE",

            "E3 Type",

            "e3_type",

        ]

    elif enzyme_class == "DUB":

        enz_ac_candidates = [

            "SwissProt AC (DUB)",

            "UniProt AC (DUB)",

            "DUB UniProt AC",

            "DUB SwissProt AC",

            "dub_uniprot_ac",

            "dub_ac",

        ]

        enz_gene_candidates = [

            "Gene Symbol (DUB)",

            "DUB Gene Symbol",

            "DUB",

            "dub_gene",

            "dub_symbol",

        ]

        enzyme_type_candidates = [

            "DUBTYPE",

            "DUB Type",

            "dub_type",

        ]

    else:

        raise ValueError("enzyme_class must be either 'E3' or 'DUB'.")

    sub_ac_candidates = [

        "SwissProt AC (Substrate)",

        "UniProt AC (Substrate)",

        "Substrate UniProt AC",

        "Substrate SwissProt AC",

        "substrate_uniprot_ac",

        "substrate_ac",

        "sub_ac",

    ]

    sub_gene_candidates = [

        "Gene Symbol (Substrate)",

        "Substrate Gene Symbol",

        "Substrate",

        "substrate_gene",

        "sub_gene",

        "substrate_symbol",

    ]

    pmid_candidates = [

        "SOURCEID",

        "PMID",

        "PubMedID",

        "PubMed ID",

        "Reference",

        "References",

    ]

    enz_ac_col = find_column(df, enz_ac_candidates)

    sub_ac_col = find_column(df, sub_ac_candidates)

    enz_gene_col = find_column(df, enz_gene_candidates)

    sub_gene_col = find_column(df, sub_gene_candidates)

    enzyme_type_col = find_column(df, enzyme_type_candidates)

    pmid_col = find_column(df, pmid_candidates)

    logging.info(f"{enzyme_class}: selected enz_ac_col = {enz_ac_col}")

    logging.info(f"{enzyme_class}: selected sub_ac_col = {sub_ac_col}")

    logging.info(f"{enzyme_class}: selected enz_gene_col = {enz_gene_col}")

    logging.info(f"{enzyme_class}: selected sub_gene_col = {sub_gene_col}")

    logging.info(f"{enzyme_class}: selected enzyme_type_col = {enzyme_type_col}")

    logging.info(f"{enzyme_class}: selected pmid_col = {pmid_col}")

    missing_required = []

    if enz_ac_col is None:

        missing_required.append("enzyme accession")

    if sub_ac_col is None:

        missing_required.append("substrate accession")

    if missing_required:

        raise ValueError(

            f"{enzyme_class}: missing required columns: {missing_required}\n"

            f"Available columns:\n{list(df.columns)}"

        )

    out = pd.DataFrame()

    out["enzyme_class"] = enzyme_class

    out["enz_ac"] = df[enz_ac_col].map(normalize_ac)

    out["sub_ac"] = df[sub_ac_col].map(normalize_ac)

    out["enz_gene"] = df[enz_gene_col].map(normalize_gene) if enz_gene_col else np.nan

    out["sub_gene"] = df[sub_gene_col].map(normalize_gene) if sub_gene_col else np.nan

    out["enzyme_type"] = df[enzyme_type_col].astype(str).str.strip() if enzyme_type_col else np.nan

    out["pmid"] = df[pmid_col].astype(str).str.strip() if pmid_col else np.nan

    out["label"] = 1

    out["source"] = "UbiBrowser2"

    # Remove missing critical identifiers

    before_drop = len(out)

    out = out.dropna(subset=["enz_ac", "sub_ac"]).copy()

    logging.info(f"{enzyme_class}: drop missing enz_ac/sub_ac: {before_drop} -> {len(out)}")

    # Stable IDs

    out["pair_id"] = (

        out["enzyme_class"].astype(str)

        + "|"

        + out["enz_ac"].astype(str)

        + "|"

        + out["sub_ac"].astype(str)

    )

    out["group_id"] = (

        out["enzyme_class"].astype(str)

        + "|"

        + out["enz_ac"].astype(str)

    )

    out = out[

        [

            "pair_id",

            "group_id",

            "enzyme_class",

            "enz_ac",

            "sub_ac",

            "enz_gene",

            "sub_gene",

            "enzyme_type",

            "label",

            "source",

            "pmid",

        ]

    ]

    return out

def merge_pmids(series: pd.Series):

    """

    Merge PMID/source IDs from duplicate rows.

    """

    values = []

    for x in series.dropna().astype(str):

        if x.lower() in {"nan", "none", "null", ""}:

            continue

        parts = re.split(r"[;,|]", x)

        values.extend([p.strip() for p in parts if p.strip()])

    values = sorted(set(values))

    return ";".join(values) if values else np.nan

def deduplicate_positive_pairs(df: pd.DataFrame) -> pd.DataFrame:

    """

    One row per pair_id.

    """

    agg = {

        "group_id": "first",

        "enzyme_class": "first",

        "enz_ac": "first",

        "sub_ac": "first",

        "enz_gene": "first",

        "sub_gene": "first",

        "enzyme_type": "first",

        "label": "first",

        "source": "first",

        "pmid": merge_pmids,

    }

    out = df.groupby("pair_id", as_index=False).agg(agg)

    out = out[

        [

            "pair_id",

            "group_id",

            "enzyme_class",

            "enz_ac",

            "sub_ac",

            "enz_gene",

            "sub_gene",

            "enzyme_type",

            "label",

            "source",

            "pmid",

        ]

    ]

    return out

def make_qc_row(df: pd.DataFrame, dataset_name: str) -> dict:

    """

    Basic QC row.

    """

    return {

        "dataset": dataset_name,

        "n_rows": len(df),

        "n_unique_pair_id": df["pair_id"].nunique(),

        "n_duplicate_pair_id_rows": int(df.duplicated("pair_id").sum()),

        "n_unique_group_id": df["group_id"].nunique(),

        "n_unique_enz_ac": df["enz_ac"].nunique(),

        "n_unique_sub_ac": df["sub_ac"].nunique(),

        "n_missing_enz_ac": int(df["enz_ac"].isna().sum()),

        "n_missing_sub_ac": int(df["sub_ac"].isna().sum()),

        "n_missing_enz_gene": int(df["enz_gene"].isna().sum()),

        "n_missing_sub_gene": int(df["sub_gene"].isna().sum()),

        "n_missing_enzyme_type": int(df["enzyme_type"].isna().sum()),

        "n_missing_pmid": int(df["pmid"].isna().sum()),

    }

# ============================================================

# Main

# ============================================================

In [ ]:
def main():

    logging.info("Starting positive parsing from raw UbiBrowser files.")

    logging.info(f"Project root: {PROJECT_ROOT}")

    logging.info(f"E3 raw path: {E3_RAW_PATH}")

    logging.info(f"DUB raw path: {DUB_RAW_PATH}")

    e3_raw = read_any_table(E3_RAW_PATH)

    dub_raw = read_any_table(DUB_RAW_PATH)

    # Save parsed raw copies for inspection

    e3_raw.to_csv(INTERIM_DIR / "e3_raw_read.csv", index=False)

    dub_raw.to_csv(INTERIM_DIR / "dub_raw_read.csv", index=False)

    # Human filtering

    e3_human = filter_human_rows(e3_raw, "E3")

    dub_human = filter_human_rows(dub_raw, "DUB")

    e3_human.to_csv(INTERIM_DIR / "e3_human_filtered.csv", index=False)

    dub_human.to_csv(INTERIM_DIR / "dub_human_filtered.csv", index=False)

    # Standardize

    e3_pos_raw_schema = standardize_positive_table(e3_human, "E3")

    dub_pos_raw_schema = standardize_positive_table(dub_human, "DUB")

    # Save before dedup

    e3_pos_raw_schema.to_csv(INTERIM_DIR / "positive_e3_before_dedup.csv", index=False)

    dub_pos_raw_schema.to_csv(INTERIM_DIR / "positive_dub_before_dedup.csv", index=False)

    # Duplicate rows for debugging

    e3_dups = e3_pos_raw_schema[

        e3_pos_raw_schema.duplicated("pair_id", keep=False)

    ].sort_values("pair_id")

    dub_dups = dub_pos_raw_schema[

        dub_pos_raw_schema.duplicated("pair_id", keep=False)

    ].sort_values("pair_id")

    e3_dups.to_csv(QC_DIR / "positive_e3_duplicate_rows.csv", index=False)

    dub_dups.to_csv(QC_DIR / "positive_dub_duplicate_rows.csv", index=False)

    # Deduplicate

    e3_pos = deduplicate_positive_pairs(e3_pos_raw_schema)

    dub_pos = deduplicate_positive_pairs(dub_pos_raw_schema)

    positive_all = pd.concat([e3_pos, dub_pos], ignore_index=True)

    # Cross-class same AC pair check

    positive_all["_pair_no_class"] = (

        positive_all["enz_ac"].astype(str)

        + "|"

        + positive_all["sub_ac"].astype(str)

    )

    cross_class = (

        positive_all.groupby("_pair_no_class")["enzyme_class"]

        .nunique()

        .reset_index(name="n_classes")

    )

    cross_class = cross_class[cross_class["n_classes"] > 1].copy()

    cross_class.to_csv(QC_DIR / "positive_cross_class_same_ac_pairs.csv", index=False)

    positive_all = positive_all.drop(columns=["_pair_no_class"])

    # E3/DUB gene overlap sanity check

    e3_genes = set(e3_pos["enz_gene"].dropna().astype(str))

    dub_genes = set(dub_pos["enz_gene"].dropna().astype(str))

    gene_overlap = sorted(e3_genes & dub_genes)

    pd.DataFrame({"overlap_gene": gene_overlap}).to_csv(

        QC_DIR / "positive_e3_dub_enzyme_gene_overlap.csv",

        index=False,

    )

    # Save final positives

    e3_pos.to_csv(PROC_DIR / "positive_e3.csv", index=False)

    dub_pos.to_csv(PROC_DIR / "positive_dub.csv", index=False)

    positive_all.to_csv(PROC_DIR / "positive_all.csv", index=False)

    # QC

    qc_rows = [

        make_qc_row(e3_pos_raw_schema, "E3_before_dedup"),

        make_qc_row(e3_pos, "E3_after_dedup"),

        make_qc_row(dub_pos_raw_schema, "DUB_before_dedup"),

        make_qc_row(dub_pos, "DUB_after_dedup"),

        make_qc_row(positive_all, "ALL_positive"),

    ]

    qc = pd.DataFrame(qc_rows)

    qc.to_csv(QC_DIR / "positive_qc.csv", index=False)

    # Console summary

    print("\n========================================")

    print("Positive parsing finished successfully")

    print("========================================")

    print(f"E3 positives:        {len(e3_pos):,}")

    print(f"DUB positives:       {len(dub_pos):,}")

    print(f"All positives:       {len(positive_all):,}")

    print(f"Unique enzymes:      {positive_all['group_id'].nunique():,}")

    print(f"Unique substrates:   {positive_all['sub_ac'].nunique():,}")

    print(f"E3 duplicate rows:   {len(e3_dups):,}")

    print(f"DUB duplicate rows:  {len(dub_dups):,}")

    print(f"Cross-class AC pairs:{len(cross_class):,}")

    print(f"E3/DUB gene overlap: {len(gene_overlap):,}")

    print("\nSaved files:")

    print(PROC_DIR / "positive_e3.csv")

    print(PROC_DIR / "positive_dub.csv")

    print(PROC_DIR / "positive_all.csv")

    print(QC_DIR / "positive_qc.csv")

    print(LOG_FILE)

    logging.info("Positive parsing finished successfully.")

if __name__ == "__main__":

    main()